### Backtest Statistics

Most of the functions below can be found under:

* Backtest/stats_measure
* sample-data/make_data

A quick slideshow: [Seven Sins of Backtesting](https://newyork.qwafafew.org/wp-content/uploads/sites/4/2015/10/Luo_20150128.pdf)

If you are keen on generating synthetic data for your research, copy the code snippets [Generate synthetic raw data](https://gist.github.com/boyboi86/5e00faf48f60abfdbe838fbdee269471) in my gist.

[Additional resource](https://en.wikipedia.org/wiki/Euler%E2%80%93Mascheroni_constant)

Contact: boyboi86@gmail.com

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cqrlib as rs

%matplotlib inline

In [ ]:
portfolio_mtx = rs.create_portfolio(price = [95, 1000], 
                                 position = [1000, 10000], 
                                 n_sample = 2000, 
                                 imbalance = True)

# this is a dividend based investor..

**Note**

The below is our entire portfolio universe, expressed in matrix format.

Sythetic portfolio assumptions:
* In this portfolio, there are 2,000 assets (Label 0 = equity, label 1 = bond).
* There can only be 2 kind of assets (You may modify the code to include more).
* All long positions, however if you are doing both long/ short style investing. Just change sign of open position.
* Guaranteed to have dividend or interest.
* Clean price are meant for bond assets, for equity just refer to open/ close price.
* Formula for bond yield:

        Accured interest = dirty close - clean close
        
* Position size will not turn from negative to positive and vice-versa (No flipping).
* Open price can be replaced with average transacted price, if you were executing average cost pricing.
* If average cost pricing method is employed, only account for single direction transaction.
* Same asset/ securities but different direction will be considered in a seperate row.
* Each position can only reflect 1 direction throughout that period.
* Datetime Index are beginning period, t1 is the ending period.
* The below are expressed in single unit of asset:
    1. Open
    2. Close
    3. Yield
    4. Clean Close
    5. Clean Open


* Junk bonds and dividend equities are expressed in percentage of entire portfolio.
* Clean and dirty price is unique to only bonds (Label = 1).
* Yield is expressed as a nominal term (The portfolio generator based on close price).
* If profit/ loss made in sale/ purchase from long/ short position will be reflected in yield.
* All yield must be positive, the function will reflect short sales.
* Expense represents all costs incurred that is due to asset during sub-period.
* Expense is expressed in nominal terms, which consist:

    1. Taxes
    2. Transaction fees
    3. Out-of-pockets
    4. Custodian charges
    5. Margin Facilities
    6. Any misc charges/ expenses
    
    
**Note**

If you are shorting instead, it will not increase your AUM but it will increase profit/ loss.

**Note**

This code snippet is not from the book. But I feel that it's necessary for me since I do not have S&P 500, to finish this tutorial.

Therefore, if you are keen to use any code snippets, please go through them before using it. 

At the same time, if you feel that you have a better way to run this algo, let me know.

In [ ]:
portfolio_mtx

In [ ]:
def rtn_by_tw(mtx: pd.DataFrame):
    df0 = pd.DataFrame(index=mtx.index).assign(t1 = mtx.t1)
    mtx['equity_diff'] = mtx['close'].sub(mtx['open'])
    mtx['bond_diff'] = mtx['clean_close'].sub(mtx['clean_open'])
    for idx in mtx.index:
        _close_pos = mtx.loc[idx, 'close_pos']
        _open_pos = mtx.loc[idx, 'open_pos']
        if ((_open_pos) > .0 and (_close_pos) >= .0) or ((_open_pos) < .0 and (_close_pos) >= .0):
            mtx.loc[idx, 't1_pos'] = max(0, float(_close_pos - _open_pos))
            mtx.loc[idx, 't1_vol'] = float(_open_pos - _close_pos)
        if ((_open_pos) > .0 and (_close_pos) <= .0) or ((_open_pos) > .0 and (_close_pos) <= .0):
            mtx.loc[idx, 't1_pos'] = max(0, float(_close_pos - _open_pos)) #shorting doesn't increase your aum
            mtx.loc[idx, 't1_vol'] = float(_open_pos - _close_pos)
            mtx[idx, 'yield'].mul(-1) #if you short; you will pay for the div/ interest instead
        
        #but it does add to your profit.. or losses
        if mtx.loc[idx, 'label'] == 0:
            mtx.loc[idx, 'pnl'] = (mtx.loc[idx, 'bond_diff'] + mtx.loc[idx, 'yield']) * mtx.loc[idx, 't1_vol']
        else:
            mtx.loc[idx, 'pnl'] = (mtx.loc[idx, 'equity_diff'] + mtx.loc[idx, 'yield']) * mtx.loc[idx, 't1_vol']
            
    
    t0_val = mtx['close'].mul(mtx['open_pos'])
    t1_val = mtx['close'].mul(mtx['t1_pos'])
    
    df0['rtn'] = mtx['pnl'].sub(mtx['expense']).div(t0_val.add(t1_val))
    df0['rtn'].add(1)

    return df0

In [ ]:
df2 = rtn_by_tw(mtx = portfolio_mtx)

df2

**Note**

Based on the above, we will have to run the next function using parallelization, so that we can link these performance geometrically and weight these returns based on time.

Otherwise, it may take much longer than expected.

If you are interested in parallelization but want some demostration.

[AFML 13.1](https://github.com/boyboi86/AFML/blob/master/AFML%2013.1.ipynb)

**Note**

Running parallel operations for binary integer is significantly faster than float.

If required try downcast dtype (Last resort).

In [ ]:
# After parallelization!

tw_rtn = rs.rtn_by_tw(mtx = portfolio_mtx, num_threads=3)

**Note**

Assuming all the asset and transaction was provided by the portfolio manager.

The below is the final outcome for the portfolio performance over her career.

If the return is negative at any time, it means the manager is having a negative return during that period.

**Hint**

Notice I only generated 2000 samples, but I have 4000 returns.

In [ ]:
# we will be using this as out back-test returns

tw_rtn

**Note**

Key high-frequency trading strategy charactertistics:
1. low return on transaction cost
2. short holding period
3. High number of bets

If we factor high leverage as part of cost (Margin facilities), each transaction will bear additional cost.

This will increase as part of it's variable cost per transaction basis, but if strategy also exhibits:

1. High turnover
2. High sharpe ratio

There is a good chance that it will have a large capacity (High profitability) even with leverage, in fact leverage can further increase it's profit ratios.

In [ ]:
df1 = pd.DataFrame({'close': tw_rtn}).resample('M').mean().fillna(0)

df1

#we have about 93 months of data

**Note**

The below ratios are calculated based on 252 trading days.

This is not correct, because we have a mixed of bonds and equities.

252 trading days are meant for equities only. But for simplicity I used 252.

Please check the number of trading days for each asset classes.

In [ ]:
positive_hhi, negative_hhi, time_hhi = rs.hh_idx(rtn = df1, freq = 'M')

sr = rs.sharpe_ratio(rtn = df1.squeeze(), rf_param = .0, t_days = 252)
ir = rs.inform_ratio(rtn = df1.squeeze(), benchmark = .0, t_days = 252)

print("\nSharpe Ratio: {0:.6f}\nInformation Ratio: {1:.6f}".format(sr, ir))

# Almost quite guarantee this portfolio manager will lose your money

### Herfindahl-Hirschman Index (HHI)

Measures the imbalance between positive and negative returns, which will reflect how skewed return distribution.

HHI time shown above indicates the frequency of these returns (concentration).

All scores return should be as low as possible.

* Low Postive HHI (No right fat tail)
* Low Negative HHI (No left fat tail)
* Low HHI Time (Return frequency is uniformly distributed)

This will impact Probabilistic Sharpe Ratio (PSR) and Deflated Sharpe Ratio (DSR). Since sampls are simulated and not actual inputs.

Therefore ideally, it should be as "normal" as possible.

In [ ]:
from scipy.stats import skew, kurtosis

pct = rs.proba_sr(obs_sr = sr,
             benchmark_sr = -1.9, # since the sr was negative to begin with
             num_returns = 93, #monthly basis
             skew_returns = df1.skew(),
             kurt_returns = df1.kurtosis())

print("The probability indicates that our sharpe ratio: {0:.6f} is not correct\nIt is {1} which is much lesser than 0.95".format(sr, pct[0]))

In [ ]:
pct = rs.proba_sr(obs_sr = sr,
             benchmark_sr = -2.15, # since the sr was negative to begin with
             num_returns = 93, #monthly basis
             skew_returns = df1.skew(),
             kurt_returns = df1.kurtosis())

print("The probability indicates that our sharpe ratio: {0:.6f} is correct\nIt is {1} more than 0.95".format(sr, pct[0]))

### Probabilistic Sharpe Ratio (PSR)

Notice the change in benchmark can reflect how correct the sharpe ratios are.

In the first example, we set benchmark as -1.9. We have a score much lesser than 0.95. (0.95 > 0.8261832625852505)

But when we change the benchmark to be below our existing sharp ratio, it became "correct".

Like-wise if we were to set a benchmark closer to the sharpe ratio, it becomes more "reliable". (0.95 < 0.9994188490783524)

### Deflated Sharpe Ratio (DSR) probability

Notice how high is the benchmark as a result deflated sharpe ratio probability was reduced. (4.980385988784981e-241 < 0.95)

If you reduce the number of trials, you will notice there will be an increase in DSR probability.

In [ ]:
std_ = 0.5
n_trials = 100

dsr, benchmark = rs.deflated_sr(obs_sr = sr,
                                sr_est = [std_, n_trials],
                                num_returns = 93,
                                skew_returns = df1.skew(), 
                                kurt_returns = df1.kurtosis(),
                                est_param = True)

print("Deflated Sharpe Ratio Probability: {0}\nBenchmark set: {1}".format(dsr[0], benchmark))

### Conclusion

When running backtest performance metrics, the following key factors will impact the reliablility of outcome.

1. Return's skewness
2. Return's Kurtosis
3. Frequency of return report
4. Risk-free rate/ benchmark

For the first 4 factors will result in deviations from normal, as a result it will mean either "too good to be true" or "too bad to be true". Where benchmark will act as a mean of return distribution.

For DSR, which corrects SR for inflationary effect due to non-normal return, track record length and selection bias. Requires multiple testing (n_trials/ number of backtest performed). As well as standard deviation of these resulted sharpe ratio.

As discussed previously overfitting is common during backtest, hence more trials performed will require higher benchmark to prevent any possible overfitting.

**Note**

In the book, Dr Marcos mentioned time weighted return calculation for a single portfolio. 

Which was what we used previously, however the remaining exercise is more directed at individual securities.

But for completeness, I load whatever S&P500 data I have for the below.

In [ ]:
df = pd.read_csv('../sample-data/dollar_bars.txt', 
                 sep=',', 
                 header=0, 
                 parse_dates = True, 
                 index_col=['date_time'])

df = df['close'].to_frame()

draw_dn, time_ddn = rs.drawdn_period(data = df, dollars = True, q_param = .95)

a_rtn = rs.annualized_rtn(data = df.squeeze(), t_days = 252, verbose = True)

**Interesting Fact**

When I tried to run the synthetic data generator to create a portofolio based on random uniform distribution.

I have never gotten once a "portfolio" with performance that has positive performance, the current portfolio in this exercise has the best "performance".

At the very least, it does not return a NaN probability.

Indirectly, it means it is almost impossible to generate alpha on random basis (Luck).